## Importa libs

In [ ]:
import pickle
from typing import Any, cast

import numpy as np
import onnxruntime as ort
import pandas as pd
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score

## Consts

In [37]:
YEAR_MONTH = "2026-04"

EDA_X_TEST_PATH = "data/x_test.parquet"
EDA_Y_TEST_PATH = "data/y_test.parquet"
EDA_MODEL_PATH = "./model/best_churn_mlp.onnx"
EDA_PREPROCESSOR_PATH = "./model/preprocessor.pkl"

PROJECT_X_TEST_PATH = f"../data/gold/{YEAR_MONTH}/x_test.parquet"
PROJECT_Y_TEST_PATH = f"../data/gold/{YEAR_MONTH}/y_test.parquet"
PROJECT_MODEL_PATH = "../models/train/churn_mlp_model.onnx"
PROJECT_PREPROCESSOR_PATH = "../models/train/preprocessor.pkl"


## Define função de evaluate

In [38]:
def evaluate_model(y_true, y_pred, y_probs) -> dict[str, float]:
    classification_rep_str = classification_report(y_true, y_pred)
    auc_roc = roc_auc_score(y_true, y_probs)
    pr_auc = average_precision_score(y_true, y_probs)

    print(classification_rep_str)
    print(f"AUC-ROC: {auc_roc:.4f}")
    print(f"PR-AUC: {pr_auc:.4f}")

    classification_rep = classification_report(y_true, y_pred, output_dict=True)
    classification_rep = cast(dict[str, Any], classification_rep)

    metrics = {
        "accuracy": classification_rep.get("accuracy", 0.0),
        "precision_weighted": classification_rep.get("weighted avg", {}).get("precision", 0.0),
        "recall_weighted": classification_rep.get("weighted avg", {}).get("recall", 0.0),
        "f1_weighted": classification_rep.get("weighted avg", {}).get("f1-score", 0.0),
        "auc_roc": auc_roc,
        "pr_auc": pr_auc,
    }
    return metrics

## Importa dados do EDA e gera metricas

In [39]:
# EDA
eda_df_x_test = pd.read_parquet(EDA_X_TEST_PATH)
eda_df_y_test = pd.read_parquet(EDA_Y_TEST_PATH)

with open(EDA_PREPROCESSOR_PATH, "rb") as f:
    eda_preprocessor = pickle.load(f)

eda_df_x_processed = eda_preprocessor.transform(eda_df_x_test).astype(np.float32)

eda_session = ort.InferenceSession(EDA_MODEL_PATH)
eda_input_name = eda_session.get_inputs()[0].name
eda_output_name = eda_session.get_outputs()[0].name
eda_resultados = eda_session.run([eda_output_name], {eda_input_name: eda_df_x_processed})

eda_y_probs = np.array(eda_resultados[0]).reshape(-1, 1)
eda_y_pred = (eda_y_probs > 0.5).astype(int)

eda_metrics = evaluate_model(eda_df_y_test.values, eda_y_pred, eda_y_probs)

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409

AUC-ROC: 0.8530
PR-AUC: 0.6526


In [40]:
# project
project_df_x_test = pd.read_parquet(PROJECT_X_TEST_PATH)
project_df_y_test = pd.read_parquet(PROJECT_Y_TEST_PATH)

with open(PROJECT_PREPROCESSOR_PATH, "rb") as f:
    project_preprocessor = pickle.load(f)

project_df_x_processed = project_preprocessor.transform(project_df_x_test).astype(np.float32)

project_session = ort.InferenceSession(PROJECT_MODEL_PATH)
project_input_name = project_session.get_inputs()[0].name
project_output_name = project_session.get_outputs()[0].name
project_resultados = project_session.run(
    [project_output_name], {project_input_name: project_df_x_processed}
)

project_y_probs = np.array(project_resultados[0]).reshape(-1, 1)
project_y_pred = (project_y_probs > 0.5).astype(int)

project_metrics = evaluate_model(project_df_y_test.values, project_y_pred, project_y_probs)

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1014
           1       0.68      0.55      0.61       395

    accuracy                           0.80      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.79      0.80      0.80      1409

AUC-ROC: 0.8610
PR-AUC: 0.7074


## Compara métricas
Compara as métricas entre o modelo gerado pelo EDA e o gerado pelo projeto

In [41]:
all_metrics = {
    "EDA": eda_metrics,
    "PROJETO": project_metrics,
}

df_comparativo = pd.DataFrame(all_metrics).T

cols = ["accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "auc_roc", "pr_auc"]
df_comparativo = df_comparativo[cols]

display(df_comparativo)

,accuracy,precision_weighted,recall_weighted,f1_weighted,auc_roc,pr_auc
EDA,0.800568,0.791503,0.800568,0.793884,0.853006,0.652553
PROJETO,0.801987,0.793672,0.801987,0.795133,0.860960,0.707384


O modelo gerado pelo projeto teve uma vantagem em relação o gerado pelo EDA. O principal motivo para isso foi o fato de que o preprocessor do EDA recebeu o conjunto de validação na hora do seu fit, pois não era utilizado essa ténica nos baselines e ele herdou isso, já o código do projeto não permitiu esse dataleakege fazendo com que o modelo tivesse de ter maior esforço na hora de treinar.